# 2. Documents

Анализ документов

In [1]:
from pydantic import Field
from src import Model

class QueryModel(Model):
    """
    Информация о запросе пользователя
    """
    settlement : str = Field(description='Населенный пункт')
    location : str = Field(description='Расположение')
    current_year : int = Field(description='Текущий год')

class StrategyModel(Model):
    """
    Результаты анализа рассматриваемого документа стратегического планирования относительно населенного пункта
    """
    goals : list[str] = Field(description='Цели документа')
    tasks : list[str] = Field(description='Задачи документа')
    problems : list[str] = Field(description='Проблемы, риски и вызовы')

class SchemaObjectModel(Model):
    """
    Описание планируемого к размещению объекта
    """
    name : str = Field(description='Наименование объекта')
    location : str = Field(description='Расположение объекта')

class SchemaModel(Model):
    """
    Результаты анализа рассматриваемой схемы территориального планирования относительно населенного пункта
    """
    objects : list[SchemaObjectModel] = Field(min_length=0, description='Планируемые к размещению объекты, относящиеся к населенному пункту')

class DocumentModel(Model):
    """
    Результаты анализа рассматриваемого документа относительно населенного пункта
    """
    objects : list[str] = Field(min_length=0, description='Планируемые к размещению объекты')

In [2]:
from typing import TypedDict, Annotated
import operator

class State(TypedDict):
    query : QueryModel
    documents : Annotated[list, operator.add]

state = {
    'query': QueryModel(
        settlement='г. Гатчина',
        location='Северо-западный федеральный округ, Ленинградская область, Гатчинский муниципальный округ (бывш. Гатчинский муниципальный район)',
        current_year=2026
    ),
    'documents': []
}

In [4]:
from src import embedding, Agent
from src.document_store import DocumentStore, DocumentSearcher

def create_node(file_path : str):
    file_name = file_path.split('/')[-1]
    doc_store = DocumentStore(embedding, file_path)
    
    def node(state : State):
        doc_searcher = DocumentSearcher(doc_store)
        agent = Agent(tools=[doc_searcher.tool], system_prompt="""
        Если 2 запроса подряд возвращается пустой результат, НЕМЕДЛЕННО формируй ответ на основе известных данных.
        """, response_format=SchemaModel)
        res = agent.run(str(state['query']))
        return {
            'documents': [res]
        }
        
    return file_name, node

In [5]:
from tqdm import tqdm
from langgraph.graph import StateGraph, END, START

graph = StateGraph(State)

for file in tqdm(files):
    n, f = create_node(file)
    graph.add_node(n, f)
    graph.add_edge(START, n)
    graph.add_edge(n, END)
    
app = graph.compile()

100%|██████████| 4/4 [00:10<00:00,  2.56s/it]


In [ ]:
from pydantic import Field
from src import Model

class FindingModel(Model):
    """
    Утверждение, тезис или вывод, результат анализа
    """
    content : str
    pages : list[int] = Field(description='Подтверждающие номера страниц исходного документа')

class GoalsModel(Model):
    """
    Цели документа
    """
    goals : list[str] = Field(min_length=0)

class TasksModel(Model):
    """
    Задачи документа
    """
    tasks : list[str] = Field(min_length=0)

class ProblemsModel(Model):
    """
    Проблемы и вызовы, рассматриваемые в документе
    """
    content : str = Field(description='Текстовое описание')

class GoalSetting(Model):
    """
    Целеполагание рассматриваемого документа
    """
    problems : str = Field(description='')

class StrategyModel(Model):
    """
    Результаты анализа документа стратегического планирования относительно населенного пункта
    """
    goals : str = Field("Цели документа")
    tasks : TasksModel = Field(description='Задачи документа')
    # problems : 

# class StrategyState(StrategyModel):


In [13]:
from src.document_store import DocumentStore, DocumentSearcher
from src import embedding, Agent

doc_store = DocumentStore(embedding, './data/documents/2. Региональный уровень/ССЭР Ленинградской области.pdf')
doc_searcher = DocumentSearcher(doc_store)
agent = Agent(tools=[doc_searcher.tool], system_prompt="""
Если 2 запроса подряд возвращается пустой результат, НЕМЕДЛЕННО формируй ответ на основе известных данных.
""", debug=True)
res = agent.run('Цели документа')

[values] {'messages': [HumanMessage(content='Цели документа', additional_kwargs={}, response_metadata={}, id='3053f7fa-4e87-479a-b4c2-0eeb46a20f4a')]}
[updates] {'model': {'messages': [AIMessage(content='Я помогу вам найти информацию о целях документа. Давайте выполним поиск по этому вопросу.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 630, 'total_tokens': 715, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 576}, 'prompt_cache_hit_tokens': 576, 'prompt_cache_miss_tokens': 54}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'd469d8ae-453b-4bcb-b9a4-fef2956d98af', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf675-bc3e-7402-bb1b-ddb595c2d9ca-0', tool_calls=[{'name': '_search', 'args': {'query': 'цели документа задачи цели', 'search_type': 'similarity'}, 'id': 'call_00_

In [14]:
print(res)

На основе найденной информации я могу сделать вывод о целях документа. Судя по контексту, это документ о стратегии развития Ленинградской области. 

**Основные цели документа:**

1. **Стратегическая цель Ленинградской области** - повышение уровня социально-экономического развития региона.

2. **Ключевые цели** включают:
   - Повышение качества среды проживания
   - Развитие промышленности, сельского хозяйства и сферы услуг
   - Повышение эффективности государственного и муниципального управления
   - Привлечение федерального ресурса и частных инвестиций
   - Улучшение инвестиционного климата в регионе

3. **Цели I этапа Стратегии** - создание условий для стабилизации и устойчивости развития региона.

4. **Конкретные задачи** направлены на:
   - Снятие социальных и экономических ограничений
   - Дополнительное материальное обеспечение молодых семей
   - Развитие социальной инфраструктуры
   - Персонализированный подход к предоставлению услуг
   - Повышение доли автомобильных дорог, соот

In [ ]:
res = app.invoke(state, debug=True)

[values] {'query': QueryModel(settlement='г. Гатчина', location='Северо-западный федеральный округ, Ленинградская область, Гатчинский муниципальный округ (бывш. Гатчинский муниципальный район)', current_year=2026), 'documents': []}
[updates] {'СТП Образование.docx': {'documents': [SchemaModel(objects=[])]}}


In [ ]:
res

{'query': QueryModel(settlement='г. Гатчина', location='Северо-западный федеральный округ, Ленинградская область, Гатчинский муниципальный округ (бывш. Гатчинский муниципальный район)', current_year=2026),
 'documents': [DocumentModel(objects=["Гатчинские ключевые болота и известняки (кластерного типа, состоит из четырех участков) (1) – 'Болото Корпиково', (2) – 'Истоки реки Парица-1', (3) – 'Истоки реки Парица-2', (4) – 'Пудость (Репузи)'", 'Карташевский ельник', 'Приоратский парк', 'Чудо-поляна']),
  DocumentModel(objects=['Лечебно-диагностический корпус с оперблоком в составе больничного городка ГБУЗ ЛО "Гатчинская КМБ" (120 коек, общая площадь не менее 3900 м²)', 'Детская поликлиника ГБУЗ ЛО "Гатчинская КМБ" (пристройка площадью не менее 900 м², мощность 100 посещений в смену)', 'Поликлиника ГБУЗ ЛО "Гатчинская КМБ" (380 посещений в смену, общая площадь не менее 3800 м²)', 'Амбулатория ГБУЗ ЛО "Гатчинская КМБ" в пос. Новый Свет (расширение до 170 посещений в смену)', 'Амбулатория Г